# 02. Предобработка и разбиение выборок

## Входы и выходы

- **Вход:** `data/global_coastal_points.parquet` (из ноутбука 01).
- **Выходы:**  
  - `data/processed_splits.npz` — массивы `X_train`, `X_val`, `X_test`, `y_*`, при необходимости список колонок признаков;  
  - `data/scaler.pkl` — `StandardScaler`, обученный **только** на train;  
  - `data/label_encoder.pkl` — соответствие имён классов и целочисленных меток.

Все модельные ноутбуки 03–10 ожидают именно эти сплиты, чтобы сравнение было на одном и том же test.

In [ ]:
from pathlib import Path
import sys
import os

for _p in (Path.cwd(), Path.cwd().parent):
    if (_p / "src" / "helpers.py").is_file():
        sys.path.insert(0, str((_p / "src").resolve()))
        os.chdir(_p)
        break

In [ ]:
from helpers import init_notebook_path, ensure_dirs, PROJECT_ROOT, PLOT_DIR, DATA_DIR, RANDOM_STATE
ROOT = init_notebook_path()
ensure_dirs()
print("PROJECT_ROOT", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from helpers import GLOBAL_DATASET_PATH, PROCESSED_PATH, feature_columns_default

## Соглашения по проекту

- Все тетради, начиная с обучающих моделей, используют `load_xy_from_processed()` и один и тот же `RANDOM_STATE` из `src/helpers`.  
- Имена классов: `CLASS_NAMES` — `OPEN_SEA`, `COASTAL_SEA`, `NEAR_COAST`, `COASTLINE` (0…3).  
- Метрики пишутся в `data/metrics_all_models.json` через `append_metrics_store(имя_модели, ...)`; сводный график — ноутбук 10.

In [ ]:
df = pd.read_parquet(GLOBAL_DATASET_PATH)
feat_cols = [c for c in feature_columns_default() if c in df.columns]
X = df[feat_cols].to_numpy(dtype=np.float64)
le = LabelEncoder()
y = le.fit_transform(df['target_class'])

print('classes', list(le.classes_))

## Разбиение и масштабирование

Используется **стратифицированное** разделение (доли и `random_state` — как в коде), чтобы в train/val/test оставались пропорции классов. Численные признаки приводятся к нулевому среднему и единичной дисперсии по **обучающей** выборке; валид. и test только трансформируются тем же `scaler`.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.85, random_state=RANDOM_STATE, stratify=y_temp)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_val = sc.transform(X_val)
X_test = sc.transform(X_test)
np.savez(PROCESSED_PATH, X_train=X_train, X_val=X_val, X_test=X_test, y_train=y_train, y_val=y_val, y_test=y_test, feature_columns=np.array(feat_cols, dtype=object))

In [ ]:
import pickle
pickle.dump(sc, open(DATA_DIR / 'scaler.pkl', 'wb'))
pickle.dump(le, open(DATA_DIR / 'label_encoder.pkl', 'wb'))
print('saved', PROCESSED_PATH, X_train.shape, X_test.shape)